In [1]:
import numpy as np
import pandas as pd

In [2]:
# reproducibility
rng = np.random.default_rng(42)

# number of players
n = 10000

In [3]:
# generate practice and skill variables (independent of each other)
practice = rng.normal(10, 4, n)         # mean, sd, num samples
practice = np.clip(practice, 0, 25)     # clip min-max

skill = rng.normal(50, 15, n)           # mean, sd, num samples
skill = np.clip(skill, 0, 100)          # clip min-max

In [4]:
# generate tournament qualification score
qualification_noise = rng.normal(0, 5, n)

qualification_score = (         # arbitrarily chosen coefficients
    2.0 * practice
    + 0.5 * skill
    + qualification_noise
)

# select top 25%, again arbitrary
qualification_threshold = np.quantile(qualification_score, 0.75)
selected = (qualification_score >= qualification_threshold).astype(int)

In [5]:
# generate performance
performance_noise = rng.normal(0, 5, n)

performance = (         # arbitrarily chosen coefficients
    30
    + 1.5 * practice
    + 0.4 * skill
    + performance_noise
)

In [6]:
# generate dataset
df = pd.DataFrame({
    "practice": practice,
    "skill": skill,
    "selected": selected,
    "performance": performance
})

df.head(10)

,practice,skill,selected,performance
0,11.218868,52.644156,0,62.828827
1,5.840064,63.486852,0,63.917849
2,13.001805,27.915941,0,53.937255
3,13.762259,45.720528,0,65.354820
4,2.195859,62.437299,0,57.047752
5,4.791282,50.984002,0,55.471097
6,10.511362,72.010007,1,78.205583
7,8.735030,26.361150,0,49.957866
8,9.932795,24.649973,0,42.409276
9,6.587824,61.545298,0,56.070991


In [11]:
# basic checks
print(df.shape)
print(df["selected"].value_counts(normalize=True).round(2))
print(df[["practice", "skill", "performance"]].describe().round(2))

(10000, 4)
selected
0    0.75
1    0.25
Name: proportion, dtype: float64
       practice     skill  performance
count  10000.00  10000.00     10000.00
mean       9.97     50.31        65.04
std        4.00     15.03         9.82
min        0.00      0.00        25.66
25%        7.29     40.06        58.41
50%        9.95     50.14        65.02
75%       12.60     60.43        71.55
max       25.00    100.00       100.44


In [12]:
# check collider effect
df_selected = df[df["selected"] == 1]

print("Practice-skill correlation, full population:")
print(f"{df['practice'].corr(df['skill']):.3f}")            # correlation practice-skill (function from pandas)

print("\nPractice-skill correlation, selected players:")
print(f"{df_selected['practice'].corr(df_selected['skill']):.3f}")

Practice-skill correlation, full population:
-0.012

Practice-skill correlation, selected players:
-0.430


In [15]:
# compare naive practice-performance slopes (naive => not accounting for bias)
full_slope = np.polyfit(            # fit a line to everyone in the dataset
    df["practice"],
    df["performance"],
    1
)[0]

selected_slope = np.polyfit(        # fit a line only to the selected players
    df_selected["practice"],
    df_selected["performance"],
    1
)[0]

print(f"True practice effect: {1.5:.3f}")
print(f"Naive slope, full population: {full_slope:.3f}")
print(f"Naive slope, selected players: {selected_slope:.3f}")

True practice effect: 1.500
Naive slope, full population: 1.479
Naive slope, selected players: 0.834


In [18]:
# save dataset
df.to_csv("collider_synthetic_data.csv",index=False,float_format="%.1f")